# FINAL IMPLEMENTATION

In [ ]:
import os
import sys
import time

import gymnasium as gym
import gymnasium_robotics
from gymnasium.wrappers import RecordVideo
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join('..')))
import config as cfg
from src.buffer import ReplayBuffer
from src.redq import REDQ
from helpers.evaluator import evaluate_policy_and_bias
from helpers.plots_video import record_agent_video, plot_experiment_results
from helpers.save_results import save_raw_logs

We will use the environment **MuJoCo Pusher** & **Box2D Bipedal Walker**.

## Sample Efficiency Comparison

The objective is to demonstrate that REDQ needs significantly fewer interactions with the environment than SAC to achieve the same level of reward.

Configuration:
* **Baseline (SAC)**: N=2, M=2, G=1.
* **REDQ**: N=10, M=2, G=20.


In [ ]:
CURRENT_EXPERIMENT = cfg.EXP_SAMPLE_EFFICIENCY
EXP_NAME = "Exp1_SampleEfficiency" 

# Iterate over ENVIRONMENTS
for current_env in cfg.ENV_NAMES:
    print("\n" + "#"*50)
    print(f"### TRAINING IN: {current_env} ###")
    print("#"*50)
    
    results_returns = {algo: np.zeros((cfg.NUM_SEEDS, cfg.EVAL_STEPS)) for algo in CURRENT_EXPERIMENT.keys()}
    results_biases = {algo: np.zeros((cfg.NUM_SEEDS, cfg.EVAL_STEPS)) for algo in CURRENT_EXPERIMENT.keys()}
    results_times = {algo: [] for algo in CURRENT_EXPERIMENT.keys()}

    # Iterate over ALGORITHMS
    for algo_name, exp_config in CURRENT_EXPERIMENT.items():
        print(f"\n{'#'*40}\n Training {algo_name} \n{'#'*40}")
        
        for seed in range(cfg.NUM_SEEDS):
            print(f"\n Seed {seed + 1}/{cfg.NUM_SEEDS}")
            start_time = time.time()
            
            # Env and agent setup
            env = gym.make(current_env)
            np.random.seed(seed)
            torch.manual_seed(seed)
            state, _ = env.reset(seed=seed)
            
            state_dim = env.observation_space.shape[0]
            action_dim = env.action_space.shape[0]
            max_action = float(env.action_space.high[0])

            replay_buffer = ReplayBuffer(state_dim, action_dim)
            agent = REDQ(state_dim, action_dim, max_action, cfg.DEVICE, 
                         num_critics=exp_config['N'], subset_size=exp_config['M'])

            eval_idx = 0

            # Training Loop
            for t in tqdm(range(cfg.MAX_TIMESTEPS), desc=f"{algo_name} (Seed {seed+1})"):
                
                # Evaluation step
                if t % cfg.EVAL_FREQ == 0:
                    ret, bias = evaluate_policy_and_bias(current_env, agent, cfg.EVAL_EPISODES)
                    results_returns[algo_name][seed, eval_idx] = ret
                    results_biases[algo_name][seed, eval_idx] = bias
                    eval_idx += 1
                
                # Interaction 
                if t < cfg.START_TIMESTEPS:
                    action = env.action_space.sample()
                else:
                    action = agent.select_action(state)
                    
                next_state, reward, terminated, truncated, _ = env.step(action)
                replay_buffer.add(state, action, reward, next_state, float(terminated))
                state = next_state
                
                # Network update
                if t >= cfg.START_TIMESTEPS:
                    agent.train(replay_buffer, cfg.BATCH_SIZE, utd_ratio=exp_config['G'])
                    
                if terminated or truncated:
                    state, _ = env.reset()

            env.close()
            
            # Time tracking
            elapsed_time = time.time() - start_time
            results_times[algo_name].append(elapsed_time)
            print(f"Time seed {seed+1}: {elapsed_time/60:.2f} minutes")

            # Video (only 1st seed)
            if seed == 0:
                record_agent_video(current_env, agent, algo_name, EXP_NAME, seed)

    # CSV store
    save_raw_logs(
        results_returns,
        results_times, 
        results_biases, 
        CURRENT_EXPERIMENT, 
        current_env, 
        cfg.MAX_TIMESTEPS, 
        cfg.EVAL_FREQ,
        EXP_NAME
    )

    # Plots
    plot_experiment_results(
        results_returns, 
        results_biases, 
        CURRENT_EXPERIMENT, 
        current_env, 
        cfg.MAX_TIMESTEPS, 
        cfg.EVAL_FREQ, 
        cfg.NUM_SEEDS,
        EXP_NAME
    )

## Ablation Study: UTD Ratio (G) Effect

**Configuration**: Fixed N=10 & M=2, varying G:

* **G=1**: REDQ working at SAC speed (but with more critics).

* **G=10**: Middle Point.

* **G=20**: Standard REDQ.

In [ ]:
CURRENT_EXPERIMENT = cfg.EXP_ABLATION_G
EXP_NAME = "Exp2_Ablation_G" 

# Iterate over ENVIRONMENTS
for current_env in cfg.ENV_NAMES:
    print("\n" + "#"*50)
    print(f"### TRAINING IN: {current_env} ###")
    print("#"*50)
    
    results_returns = {algo: np.zeros((cfg.NUM_SEEDS, cfg.EVAL_STEPS)) for algo in CURRENT_EXPERIMENT.keys()}
    results_biases = {algo: np.zeros((cfg.NUM_SEEDS, cfg.EVAL_STEPS)) for algo in CURRENT_EXPERIMENT.keys()}
    results_times = {algo: [] for algo in CURRENT_EXPERIMENT.keys()}

    # Iterate over ALGORITHMS
    for algo_name, exp_config in CURRENT_EXPERIMENT.items():
        print(f"\n{'#'*40}\n Training {algo_name} \n{'#'*40}")
        
        for seed in range(cfg.NUM_SEEDS):
            print(f"\n 🌱 Seed {seed + 1}/{cfg.NUM_SEEDS}")
            start_time = time.time()
            
            # Env and agent setup
            env = gym.make(current_env)
            np.random.seed(seed)
            torch.manual_seed(seed)
            state, _ = env.reset(seed=seed)
            
            state_dim = env.observation_space.shape[0]
            action_dim = env.action_space.shape[0]
            max_action = float(env.action_space.high[0])

            replay_buffer = ReplayBuffer(state_dim, action_dim)
            agent = REDQ(state_dim, action_dim, max_action, cfg.DEVICE, 
                         num_critics=exp_config['N'], subset_size=exp_config['M'])

            eval_idx = 0

            # Training Loop
            for t in tqdm(range(cfg.MAX_TIMESTEPS), desc=f"{algo_name} (Seed {seed+1})"):
                
                # Evaluation step
                if t % cfg.EVAL_FREQ == 0:
                    ret, bias = evaluate_policy_and_bias(current_env, agent, cfg.EVAL_EPISODES)
                    results_returns[algo_name][seed, eval_idx] = ret
                    results_biases[algo_name][seed, eval_idx] = bias
                    eval_idx += 1
                
                # Interaction 
                if t < cfg.START_TIMESTEPS:
                    action = env.action_space.sample()
                else:
                    action = agent.select_action(state)
                    
                next_state, reward, terminated, truncated, _ = env.step(action)
                replay_buffer.add(state, action, reward, next_state, float(terminated))
                state = next_state
                
                # Network update
                if t >= cfg.START_TIMESTEPS:
                    agent.train(replay_buffer, cfg.BATCH_SIZE, utd_ratio=exp_config['G'])
                    
                if terminated or truncated:
                    state, _ = env.reset()

            env.close()
            
            # Time tracking
            elapsed_time = time.time() - start_time
            results_times[algo_name].append(elapsed_time)
            print(f"Time seed {seed+1}: {elapsed_time/60:.2f} minutes")

            # Video (only 1st seed)
            if seed == 0:
                record_agent_video(current_env, agent, algo_name, EXP_NAME, seed)

    # CSV store
    save_raw_logs(
        results_returns,
        results_times, 
        results_biases, 
        CURRENT_EXPERIMENT, 
        current_env, 
        cfg.MAX_TIMESTEPS, 
        cfg.EVAL_FREQ,
        EXP_NAME
    )

    # Plots
    plot_experiment_results(
        results_returns, 
        results_biases, 
        CURRENT_EXPERIMENT, 
        current_env, 
        cfg.MAX_TIMESTEPS, 
        cfg.EVAL_FREQ, 
        cfg.NUM_SEEDS,
        EXP_NAME
    )

## Robustness and Stability: The Ensemble (N)
This experiment is key for high-dimensional environments like Pusher.

Configuration: Keep G=20, M=2 and vary N:
* N=2
* N=10

In [ ]:
CURRENT_EXPERIMENT = cfg.EXP_ROBUSTNESS_N
EXP_NAME = "Exp3_Robustness_N" 

# Iterate over ENVIRONMENTS
for current_env in cfg.ENV_NAMES:
    print("\n" + "#"*50)
    print(f"### TRAINING IN: {current_env} ###")
    print("#"*50)
    
    results_returns = {algo: np.zeros((cfg.NUM_SEEDS, cfg.EVAL_STEPS)) for algo in CURRENT_EXPERIMENT.keys()}
    results_biases = {algo: np.zeros((cfg.NUM_SEEDS, cfg.EVAL_STEPS)) for algo in CURRENT_EXPERIMENT.keys()}
    results_times = {algo: [] for algo in CURRENT_EXPERIMENT.keys()}

    # Iterate over ALGORITHMS
    for algo_name, exp_config in CURRENT_EXPERIMENT.items():
        print(f"\n{'#'*40}\n Training {algo_name} \n{'#'*40}")
        
        for seed in range(cfg.NUM_SEEDS):
            print(f"\n 🌱 Seed {seed + 1}/{cfg.NUM_SEEDS}")
            start_time = time.time()
            
            # Env and agent setup
            env = gym.make(current_env)
            np.random.seed(seed)
            torch.manual_seed(seed)
            state, _ = env.reset(seed=seed)
            
            state_dim = env.observation_space.shape[0]
            action_dim = env.action_space.shape[0]
            max_action = float(env.action_space.high[0])

            replay_buffer = ReplayBuffer(state_dim, action_dim)
            agent = REDQ(state_dim, action_dim, max_action, cfg.DEVICE, 
                         num_critics=exp_config['N'], subset_size=exp_config['M'])

            eval_idx = 0

            # Training Loop
            for t in tqdm(range(cfg.MAX_TIMESTEPS), desc=f"{algo_name} (Seed {seed+1})"):
                
                # Evaluation step
                if t % cfg.EVAL_FREQ == 0:
                    ret, bias = evaluate_policy_and_bias(current_env, agent, cfg.EVAL_EPISODES)
                    results_returns[algo_name][seed, eval_idx] = ret
                    results_biases[algo_name][seed, eval_idx] = bias
                    eval_idx += 1
                
                # Interaction 
                if t < cfg.START_TIMESTEPS:
                    action = env.action_space.sample()
                else:
                    action = agent.select_action(state)
                    
                next_state, reward, terminated, truncated, _ = env.step(action)
                replay_buffer.add(state, action, reward, next_state, float(terminated))
                state = next_state
                
                # Network update
                if t >= cfg.START_TIMESTEPS:
                    agent.train(replay_buffer, cfg.BATCH_SIZE, utd_ratio=exp_config['G'])
                    
                if terminated or truncated:
                    state, _ = env.reset()

            env.close()
            
            # Time tracking
            elapsed_time = time.time() - start_time
            results_times[algo_name].append(elapsed_time)
            print(f"Time seed {seed+1}: {elapsed_time/60:.2f} minutes")

            # Video (only 1st seed)
            if seed == 0:
                record_agent_video(current_env, agent, algo_name, EXP_NAME, seed)

    # CSV store
    save_raw_logs(
        results_returns,
        results_times, 
        results_biases, 
        CURRENT_EXPERIMENT, 
        current_env, 
        cfg.MAX_TIMESTEPS, 
        cfg.EVAL_FREQ,
        EXP_NAME
    )

    # Plots
    plot_experiment_results(
        results_returns, 
        results_biases, 
        CURRENT_EXPERIMENT, 
        current_env, 
        cfg.MAX_TIMESTEPS, 
        cfg.EVAL_FREQ, 
        cfg.NUM_SEEDS,
        EXP_NAME
    )

## Overestimation Analysis: The Subset (M)
REDQ uses the minimum of a random subset of critics to avoid being too optimistic.

**Configuration**: Keep N=10, G=20 and vary M:
* M = 2 (standard)
* M = 5 (more conservative)

In [ ]:
CURRENT_EXPERIMENT = cfg.EXP_OVERESTIMATION_M
EXP_NAME = "Exp4_Overestimation_M" 

# Iterate over ENVIRONMENTS
for current_env in cfg.ENV_NAMES:
    print("\n" + "#"*50)
    print(f"### TRAINING IN: {current_env} ###")
    print("#"*50)
    
    results_returns = {algo: np.zeros((cfg.NUM_SEEDS, cfg.EVAL_STEPS)) for algo in CURRENT_EXPERIMENT.keys()}
    results_biases = {algo: np.zeros((cfg.NUM_SEEDS, cfg.EVAL_STEPS)) for algo in CURRENT_EXPERIMENT.keys()}
    results_times = {algo: [] for algo in CURRENT_EXPERIMENT.keys()}

    # Iterate over ALGORITHMS
    for algo_name, exp_config in CURRENT_EXPERIMENT.items():
        print(f"\n{'#'*40}\n Training {algo_name} \n{'#'*40}")
        
        for seed in range(cfg.NUM_SEEDS):
            print(f"\n 🌱 Seed {seed + 1}/{cfg.NUM_SEEDS}")
            start_time = time.time()
            
            # Env and agent setup
            env = gym.make(current_env)
            np.random.seed(seed)
            torch.manual_seed(seed)
            state, _ = env.reset(seed=seed)
            
            state_dim = env.observation_space.shape[0]
            action_dim = env.action_space.shape[0]
            max_action = float(env.action_space.high[0])

            replay_buffer = ReplayBuffer(state_dim, action_dim)
            agent = REDQ(state_dim, action_dim, max_action, cfg.DEVICE, 
                         num_critics=exp_config['N'], subset_size=exp_config['M'])

            eval_idx = 0

            # Training Loop
            for t in tqdm(range(cfg.MAX_TIMESTEPS), desc=f"{algo_name} (Seed {seed+1})"):
                
                # Evaluation step
                if t % cfg.EVAL_FREQ == 0:
                    ret, bias = evaluate_policy_and_bias(current_env, agent, cfg.EVAL_EPISODES)
                    results_returns[algo_name][seed, eval_idx] = ret
                    results_biases[algo_name][seed, eval_idx] = bias
                    eval_idx += 1
                
                # Interaction 
                if t < cfg.START_TIMESTEPS:
                    action = env.action_space.sample()
                else:
                    action = agent.select_action(state)
                    
                next_state, reward, terminated, truncated, _ = env.step(action)
                replay_buffer.add(state, action, reward, next_state, float(terminated))
                state = next_state
                
                # Network update
                if t >= cfg.START_TIMESTEPS:
                    agent.train(replay_buffer, cfg.BATCH_SIZE, utd_ratio=exp_config['G'])
                    
                if terminated or truncated:
                    state, _ = env.reset()

            env.close()
            
            # Time tracking
            elapsed_time = time.time() - start_time
            results_times[algo_name].append(elapsed_time)
            print(f"Time seed {seed+1}: {elapsed_time/60:.2f} minutes")

            # Video (only 1st seed)
            if seed == 0:
                record_agent_video(current_env, agent, algo_name, EXP_NAME, seed)

    # CSV store
    save_raw_logs(
        results_returns,
        results_times, 
        results_biases, 
        CURRENT_EXPERIMENT, 
        current_env, 
        cfg.MAX_TIMESTEPS, 
        cfg.EVAL_FREQ,
        EXP_NAME
    )

    # Plots
    plot_experiment_results(
        results_returns, 
        results_biases, 
        CURRENT_EXPERIMENT, 
        current_env, 
        cfg.MAX_TIMESTEPS, 
        cfg.EVAL_FREQ, 
        cfg.NUM_SEEDS,
        EXP_NAME
    )